<a href="https://colab.research.google.com/github/amahendra/Intelligent-Systems-Engineering/blob/Notebooks/session_05_enterprise_system_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Sesi 5 — Integrasi Sistem Cerdas dengan Sistem Enterprise
**MT25C02 · Rekayasa Sistem Cerdas · Google Colab Free**

**Proyek yang sama dengan Sesi 4:** AI Assistant pemeliharaan bearing kereta `TR-001`.

**Target pembelajaran:** memahami bagaimana data sensor masuk ke aplikasi, dibaca melalui REST API, digabungkan dengan SOP dan model AI, lalu diproses sebagai event. Seluruh sistem enterprise disimulasikan dalam satu runtime Colab.

**Tidak memerlukan akun Hugging Face, API key, server eksternal, atau GPU.** Model Qwen2.5 0.5B Instruct diunduh melalui Ollama apabila Anda menjalankan bagian AI. Model kecil dapat membuat kesalahan sehingga semua rekomendasi harus diperiksa.

> **Keselamatan:** Dataset, sensor, dan sistem persetujuan adalah simulasi pendidikan. Ketiga SOP **tidak** menetapkan batas suhu keselamatan universal. Jangan menyimpulkan 92°C aman atau berbahaya tanpa standar resmi untuk aset tersebut.

## Petunjuk singkat
1. Buka `.ipynb` di Google Colab → **Runtime → Run all** (CPU sudah cukup; instalasi Ollama memerlukan internet sekali untuk unduh model).
2. Notebook **otomatis membuat** `sample_data/` berisi tiga SOP asli Sesi 4 dan `sensor_readings.csv`. Tidak perlu unggah file untuk mulai.
3. Jika ingin menggunakan tiga SOP versi Anda sendiri, ganti file `.txt` di `/content/sample_data/` sebelum menjalankan bagian pembacaan SOP.
4. Baca **Apa yang diamati?** setelah tiap eksperimen; ubah parameter yang ditunjukkan lalu jalankan ulang sel terkait.
5. Di bagian akhir, unduh CSV untuk Excel. Semua waktu sensor merupakan timestamp historis buatan.

| Eksperimen | Fokus | Keluaran |
|---|---|---|
| 1 | Data ingestion + API | JSON, status HTTP, data valid/invalid |
| 2 | Orchestration + SOP + model | Evidence, jawaban AI, state dan audit |
| 3 | Event-driven + idempotency + approval | Event log, review draft, persetujuan manual |

## 0. Persiapan Python dan folder kerja

Bagian ini hanya menginstal paket yang diperlukan. Kita memakai **FastAPI `TestClient`**, sehingga API dapat diuji menggunakan HTTP request tanpa membuka port server publik. `pandas` dipakai untuk membaca CSV.

In [ ]:
import sys, subprocess, os, json, time, re, asyncio, uuid
from pathlib import Path
from datetime import datetime, timezone

# Instal dependensi Python secara ringan dan otomatis di Colab.
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "fastapi>=0.110,<1", "httpx>=0.26,<1", "pandas>=2,<3"], check=True)
import pandas as pd
from fastapi import FastAPI, HTTPException, Header
from fastapi.testclient import TestClient

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA = ROOT / 'sample_data'
OUT = ROOT / 'hasil_sesi5'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('Folder data:', DATA)
print('Folder hasil:', OUT)

Folder data: /content/sample_data
Folder hasil: /content/hasil_sesi5


## 1. Dataset yang sama dengan Sesi 4

Tiga SOP berikut **sama persis** dengan file praktikum Sesi 4 (`SOP-PS-001`, `SOP-PS-002`, `SOP-PS-003`). Notebook menuliskannya jika belum ada. **Jika file dengan nama yang sama sudah ada, tidak ditimpa.**

Data sensor berupa CSV simulasi, dengan kolom: `event_id`, `asset_id`, `temperature_c`, `vibration_mm_s`, `timestamp`, `unit`, `source`. Nilai 92°C sengaja dipertahankan agar contoh tetap konsisten dengan Sesi 4.

In [ ]:
# Isi dokumen SOP diambil dari paket sample_data Sesi 4.
# Jangan menafsirkan dokumen ini sebagai standar keselamatan perkeretaapian nyata.
SOP_TEXTS = {'SOP-PS-001_Pemeriksaan_Awal_Bearing.txt': 'PEDOMAN KERJA PEMELIHARAAN SARANA\nNOMOR DOKUMEN : SOP-PS-001\nJUDUL        : Pemeriksaan Awal Bantalan Gandar (Axle Bearing)\nUNIT         : Pemeliharaan Sarana\nREVISI       : 00\n\n1. TUJUAN\nMenetapkan urutan pemeriksaan awal bantalan gandar agar temuan kondisi tidak normal dicatat, diverifikasi, dan ditindaklanjuti oleh petugas berwenang.\n\n2. RUANG LINGKUP\nPemeriksaan visual dan pemeriksaan kondisi awal bantalan gandar ketika sarana berada pada lokasi pemeriksaan yang telah dinyatakan aman oleh petugas lapangan.\n\n3. TANGGUNG JAWAB\n3.1 Teknisi melakukan pemeriksaan, dokumentasi, dan pelaporan temuan.\n3.2 Pengawas pemeliharaan memverifikasi temuan dan menentukan tindak lanjut melalui prosedur yang berlaku.\n3.3 Petugas keselamatan memastikan persyaratan pengamanan pekerjaan dipenuhi sesuai aturan unit kerja.\n\n4. PERSIAPAN\n4.1 Konfirmasi identitas sarana dan nomor aset dari registri aset.\n4.2 Periksa work order, riwayat pemeliharaan, dan catatan gangguan yang tersedia.\n4.3 Pastikan izin kerja, pengamanan sarana, dan penggunaan APD telah diverifikasi oleh pihak berwenang.\n4.4 Gunakan daftar periksa yang berlaku pada jenis sarana terkait.\n\n5. LANGKAH PEMERIKSAAN\n5.1 Cocokkan identitas bantalan gandar dengan data aset dan catatan pemeliharaan.\n5.2 Amati kondisi bagian yang dapat diperiksa secara aman, termasuk indikasi kebocoran, kontaminasi, kerusakan fisik, dan tanda perubahan warna yang tidak lazim.\n5.3 Catat bunyi, getaran, atau indikasi lain yang dilaporkan petugas operasi. Lakukan pengukuran tambahan hanya sesuai instruksi kerja yang berlaku.\n5.4 Bila ditemukan indikasi tidak normal, dokumentasikan lokasi, waktu, identitas aset, foto jika diizinkan, serta uraian observasi.\n5.5 Laporkan temuan kepada pengawas untuk verifikasi dan penentuan tindakan. Jangan menyimpulkan kelayakan operasi hanya dari satu indikator.\n\n6. PENCATATAN\nCatatan minimum meliputi tanggal, nomor aset, identitas pemeriksa, observasi, sumber informasi, dan nomor work order bila ada.\n\n7. ESKALASI\nTemuan yang berpotensi memengaruhi keselamatan harus segera disampaikan kepada pengawas sesuai jalur eskalasi yang berlaku. Keputusan operasional berada pada petugas yang memiliki kewenangan.\n', 'SOP-PS-002_Pemantauan_Suhu_dan_Getaran.txt': 'PEDOMAN KERJA PEMELIHARAAN SARANA\nNOMOR DOKUMEN : SOP-PS-002\nJUDUL        : Pemantauan Suhu dan Getaran Bantalan Gandar\nUNIT         : Pemeliharaan Sarana\nREVISI       : 00\n\n1. TUJUAN\nMemberikan pedoman pencatatan dan verifikasi pengamatan suhu serta getaran bantalan gandar dengan data yang dapat ditelusuri.\n\n2. RUANG LINGKUP\nPemantauan tren dan pemeriksaan data hasil sensor, inspeksi lapangan, atau perangkat pengukuran yang disetujui unit pemeliharaan.\n\n3. TANGGUNG JAWAB\n3.1 Teknisi memastikan identitas aset, waktu pengukuran, dan kondisi pengambilan data dicatat.\n3.2 Analis atau petugas teknis membandingkan observasi dengan riwayat dan kriteria resmi yang berlaku untuk tipe aset.\n3.3 Pengawas menetapkan kebutuhan pemeriksaan tambahan dan persetujuan tindak lanjut.\n\n4. PERSIAPAN DATA\n4.1 Verifikasi status kalibrasi dan kelayakan instrumen pengukuran.\n4.2 Catat identitas sarana, posisi bantalan, timestamp, kondisi operasi, dan sumber sensor.\n4.3 Pastikan satuan pengukuran serta rentang pengukuran konsisten.\n\n5. PEMANTAUAN\n5.1 Tinjau kualitas data: data hilang, duplikasi, sensor tidak aktif, dan perubahan mendadak yang berpotensi disebabkan gangguan sensor.\n5.2 Bandingkan nilai pengamatan dengan riwayat aset dan dokumen ambang resmi yang berlaku. Dokumen ini tidak menetapkan batas suhu atau getaran universal.\n5.3 Bila terdapat penyimpangan, lakukan verifikasi silang terhadap informasi operasi, riwayat gangguan, dan kondisi inspeksi.\n5.4 Catat hasil pengukuran dan analisis tren sebelum mengusulkan tindakan.\n5.5 Jangan menggunakan output model prediktif sebagai satu-satunya dasar keputusan kelayakan operasi.\n\n6. PELAPORAN\nLaporan memuat sumber data, kualitas data, hasil pengamatan, interpretasi awal, keterbatasan data, dan rekomendasi verifikasi.\n\n7. ESKALASI\nIndikasi anomali yang belum dapat dijelaskan harus diteruskan kepada pengawas. Penetapan tindakan dan status operasional mengikuti instruksi kerja, otorisasi, dan standar keselamatan yang berlaku.\n', 'SOP-PS-003_Pelaporan_dan_Persetujuan_Tindak_Lanjut.txt': 'PEDOMAN KERJA PEMELIHARAAN SARANA\nNOMOR DOKUMEN : SOP-PS-003\nJUDUL        : Pelaporan Temuan dan Persetujuan Tindak Lanjut\nUNIT         : Pemeliharaan Sarana\nREVISI       : 00\n\n1. TUJUAN\nMemastikan temuan pemeriksaan, rekomendasi teknis, dan keputusan tindak lanjut dicatat dan memiliki penanggung jawab yang jelas.\n\n2. RUANG LINGKUP\nPelaporan temuan pada aset sarana, verifikasi oleh pengawas, dan pengelolaan work order pada sistem pemeliharaan.\n\n3. PIHAK TERKAIT\n3.1 Teknisi: membuat laporan inspeksi, melampirkan bukti, dan mengusulkan pemeriksaan lanjutan.\n3.2 Supervisor: memverifikasi kelengkapan laporan, menilai prioritas, dan memberi persetujuan sesuai kewenangan.\n3.3 Administrator sistem: menjaga keutuhan catatan, akses, dan audit trail.\n\n4. PENYUSUNAN LAPORAN\n4.1 Cantumkan nomor aset, posisi komponen, tanggal, waktu, dan identitas petugas.\n4.2 Uraikan observasi faktual dan bedakan hasil ukur, indikasi model, serta dugaan yang belum terverifikasi.\n4.3 Cantumkan rujukan dokumen teknis yang digunakan.\n4.4 Sertakan foto atau data sensor bila akses dan kebijakan informasi mengizinkan.\n\n5. PROSES PERSETUJUAN\n5.1 Sistem atau teknisi mengirim laporan kepada supervisor melalui jalur kerja resmi.\n5.2 Supervisor memeriksa data, permintaan klarifikasi, dan konsekuensi keselamatan sebelum memberi keputusan.\n5.3 Rekomendasi yang dihasilkan oleh aplikasi AI diperlakukan sebagai bahan bantu, bukan persetujuan otomatis.\n5.4 Keputusan dapat berupa meminta data tambahan, menjadwalkan pemeriksaan, atau meneruskan eskalasi sesuai wewenang.\n5.5 Work order dibuat atau diperbarui oleh pengguna atau sistem yang telah mendapat otorisasi.\n\n6. AUDIT TRAIL\nCatat waktu pengajuan, identitas pengusul, perubahan data, identitas pemberi persetujuan, status tindak lanjut, dan rujukan work order.\n\n7. PENUTUP\nKeputusan operasional harus dapat ditelusuri ke bukti pemeriksaan, dokumen resmi yang berlaku, serta otorisasi manusia yang kompeten.\n'}
for filename, text in SOP_TEXTS.items():
    destination = DATA / filename
    if not destination.exists():
        destination.write_text(text, encoding='utf-8')

# Dataset sensor historis/simulasi, bukan pembacaan perangkat sungguhan.
readings = [
    {"event_id":"evt-1001", "asset_id":"TR-001", "temperature_c":89.0, "vibration_mm_s":3.1, "timestamp":"2026-11-11T09:55:00+07:00", "unit":"C", "source":"simulated-sensor"},
    {"event_id":"evt-1002", "asset_id":"TR-001", "temperature_c":92.0, "vibration_mm_s":3.8, "timestamp":"2026-11-11T10:00:00+07:00", "unit":"C", "source":"simulated-sensor"},
    {"event_id":"evt-2001", "asset_id":"TR-002", "temperature_c":56.0, "vibration_mm_s":1.2, "timestamp":"2026-11-11T10:01:00+07:00", "unit":"C", "source":"simulated-sensor"},
    {"event_id":"evt-bad",  "asset_id":"TR-001", "temperature_c":"tidak ada", "vibration_mm_s":2.0, "timestamp":"2026-11-11T10:03:00+07:00", "unit":"C", "source":"simulated-sensor"},
]
CSV_FILE = DATA / 'sensor_readings.csv'
if not CSV_FILE.exists():
    pd.DataFrame(readings).to_csv(CSV_FILE, index=False)
print('SOP tersedia:', [p.name for p in sorted(DATA.glob('*.txt'))])
print('Dataset sensor:')
display(pd.read_csv(CSV_FILE))

SOP tersedia: ['SOP-PS-001_Pemeriksaan_Awal_Bearing.txt', 'SOP-PS-002_Pemantauan_Suhu_dan_Getaran.txt', 'SOP-PS-003_Pelaporan_dan_Persetujuan_Tindak_Lanjut.txt']
Dataset sensor:


,event_id,asset_id,temperature_c,vibration_mm_s,timestamp,unit,source
0,evt-1001,TR-001,89.0,3.1,2026-11-11T09:55:00+07:00,C,simulated-sensor
1,evt-1002,TR-001,92.0,3.8,2026-11-11T10:00:00+07:00,C,simulated-sensor
2,evt-2001,TR-002,56.0,1.2,2026-11-11T10:01:00+07:00,C,simulated-sensor
3,evt-bad,TR-001,tidak ada,2.0,2026-11-11T10:03:00+07:00,C,simulated-sensor


## Eksperimen 1 — Data Ingestion dan REST API

### 2. Validasi dan normalisasi data sensor

**Tujuan:** membedakan pembacaan sensor yang dapat dipakai dan baris yang harus ditolak.

Kita memeriksa `asset_id`, tipe angka, satuan, dan `timestamp`. Dalam contoh ini tidak ada aturan untuk menilai apakah suhu tertentu aman. Validasi **format data** berbeda dari **penilaian keselamatan**.

In [ ]:
REQUIRED = {'event_id','asset_id','temperature_c','vibration_mm_s','timestamp','unit','source'}
ASSET_RE = re.compile(r'^TR-\d{3}$')

def validate_sensor(row):
    """Validasi satu pengukuran; kembalikan (data_bersih, pesan_error)."""
    if not REQUIRED.issubset(row):
        return None, 'MISSING_FIELD'
    if not ASSET_RE.fullmatch(str(row['asset_id'])):
        return None, 'INVALID_ASSET_ID'
    if row['unit'] != 'C':
        return None, 'INVALID_UNIT'
    try:
        temp = float(row['temperature_c'])
        vibration = float(row['vibration_mm_s'])
        if not (-100 <= temp <= 300) or vibration < 0:
            return None, 'OUT_OF_PLAUSIBILITY_RANGE'  # Cek kewajaran data, BUKAN batas keselamatan!
        timestamp = datetime.fromisoformat(str(row['timestamp']))
        if timestamp.tzinfo is None:
            return None, 'TIMESTAMP_NO_TIMEZONE'
    except (TypeError, ValueError, OverflowError):
        return None, 'INVALID_TYPE_OR_TIMESTAMP'
    clean = dict(row)
    clean.update(temperature_c=temp, vibration_mm_s=vibration,
                 timestamp=timestamp.isoformat(), unit='C')
    return clean, None

raw = pd.read_csv(CSV_FILE).to_dict('records')
valid, rejected = [], []
for row in raw:
    item, error = validate_sensor(row)
    (valid if error is None else rejected).append(item if error is None else
                                                  {'event_id':row.get('event_id'),'error_code':error})
print('Valid:',len(valid),'| Ditolak:',len(rejected))
display(pd.DataFrame(valid))
display(pd.DataFrame(rejected))

Valid: 3 | Ditolak: 1


,event_id,asset_id,temperature_c,vibration_mm_s,timestamp,unit,source
0,evt-1001,TR-001,89.0,3.1,2026-11-11T09:55:00+07:00,C,simulated-sensor
1,evt-1002,TR-001,92.0,3.8,2026-11-11T10:00:00+07:00,C,simulated-sensor
2,evt-2001,TR-002,56.0,1.2,2026-11-11T10:01:00+07:00,C,simulated-sensor


,event_id,error_code
0,evt-bad,INVALID_TYPE_OR_TIMESTAMP


### 3. Membuat REST API lokal

Endpoint: `GET /assets/{asset_id}/sensors/latest`.

API menyediakan **pengukuran paling akhir dalam dataset**, bukan nilai sensor real-time. Contoh ini memakai header `x-user` untuk **simulasi** otorisasi. Pada produksi, identitas harus diverifikasi menggunakan mekanisme autentikasi yang sesungguhnya.

Kode 200 = berhasil; 400 = format ID salah; 403 = akses ditolak; 404 = aset tak ditemukan; 503 = data sensor tidak tersedia.

In [ ]:
# Simulasi registri aset dan hak akses. Ini BUKAN sistem identitas produksi.
ASSETS = {'TR-001':{'component':'bearing','location':'depot A'},
          'TR-002':{'component':'bearing','location':'depot B'}}
PERMISSIONS = {'teknisi_a':{'TR-001'}, 'supervisor_a':{'TR-001','TR-002','TR-999'}}
app = FastAPI(title='Simulated Railway Sensor API')

@app.get('/assets/{asset_id}/sensors/latest')
def sensor_latest(asset_id: str, x_user: str = Header(default='')):
    # Urutan validasi: format → izin → registri → ketersediaan pembacaan.
    if not ASSET_RE.fullmatch(asset_id):
        raise HTTPException(400, detail='INVALID_ASSET_ID')
    if asset_id not in PERMISSIONS.get(x_user, set()):
        raise HTTPException(403, detail='FORBIDDEN')
    if asset_id not in ASSETS:
        raise HTTPException(404, detail='ASSET_NOT_FOUND')
    rows = [r for r in valid if r['asset_id']==asset_id]
    if not rows:
        raise HTTPException(503, detail='SENSOR_UNAVAILABLE')
    latest = max(rows, key=lambda r: r['timestamp'])
    return {'status':'success','reading':latest,'data_freshness':'historical_simulation'}

client = TestClient(app)
scenarios = [('TR-001','teknisi_a'),('ABC','teknisi_a'),('TR-002','teknisi_a'),('TR-999','supervisor_a')]
api_trials = []
for asset_id,user in scenarios:
    res=client.get(f'/assets/{asset_id}/sensors/latest',headers={'x-user':user})
    api_trials.append({'asset_id':asset_id,'user':user,'http_status':res.status_code,
                       'response':json.dumps(res.json(),ensure_ascii=False)})
display(pd.DataFrame(api_trials))

,asset_id,user,http_status,response
0,TR-001,teknisi_a,200,"{""status"": ""success"", ""reading"": {""event_id"": ..."
1,ABC,teknisi_a,400,"{""detail"": ""INVALID_ASSET_ID""}"
2,TR-002,teknisi_a,403,"{""detail"": ""FORBIDDEN""}"
3,TR-999,supervisor_a,404,"{""detail"": ""ASSET_NOT_FOUND""}"


**Apa yang diamati?** Mengapa `TR-002` menghasilkan 403 untuk `teknisi_a`? Coba ganti pengguna menjadi `supervisor_a`. Perhatikan bahwa HTTP 200 hanya membuktikan permintaan berhasil, bukan bahwa data sensor terbaru secara fisik.

## Eksperimen 2 — API Orchestration, Retrieval, dan Model AI

### 4. Baca SOP, cari dokumen, dan susun konteks

Sekarang kita menggabungkan API sensor dari eksperimen 1 dan tiga SOP yang sama seperti Sesi 4.

Agar mudah diikuti, retrieval menggunakan pencocokan kata kunci; tidak ada embedding, vector database, atau API eksternal. Metadata status aktif berikut **hanya status simulasi untuk praktikum**; sistem produksi harus mengambil versi dan status resmi dari Document Management.

In [ ]:
# Kita memuat file .txt yang telah tersedia di sample_data.
sop_docs = {}
for path in sorted(DATA.glob('*.txt')):
    text = path.read_text(encoding='utf-8')
    doc_id = re.search(r'NOMOR DOKUMEN\s*:\s*(\S+)',text)
    revision = re.search(r'REVISI\s*:\s*(\S+)',text)
    if doc_id:
        sop_docs[doc_id.group(1)] = {'text':text,'filename':path.name,
                                      'revision':revision.group(1) if revision else '?',
                                      'status':'simulasi_aktif'}
print('Dokumen:',list(sop_docs))

# Kata kunci mengikuti tujuan praktikum; diperiksa secara eksplisit agar mudah dipahami.
TOPIC_WORDS = {'SOP-PS-001':['awal','visual','inspeksi'],
               'SOP-PS-002':['suhu','temperatur','getaran','sensor','bearing'],
               'SOP-PS-003':['lapor','persetujuan','supervisor','menyetujui','tindakan','work order','tindak lanjut']}

def retrieve_sops(query, top_k=2):
    """Pilih SOP dengan skor overlap istilah; skor bukan jaminan kecukupan bukti."""
    query = query.lower()
    ranked=[]
    for doc_id,doc in sop_docs.items():
        score=sum(1 for word in TOPIC_WORDS.get(doc_id,[]) if word in query)
        if score > 0 and doc['status']=='simulasi_aktif':
            ranked.append((score,doc_id))
    ranked.sort(key=lambda x:(-x[0],x[1]))
    return [{'doc_id':doc_id,'score':score,'revision':sop_docs[doc_id]['revision'],
             'source':sop_docs[doc_id]['filename']} for score,doc_id in ranked[:top_k]]

QUESTION='Berapa suhu bearing TR-001, apa yang bisa disimpulkan, dan siapa menyetujui tindakan?'
found=retrieve_sops(QUESTION, top_k=3)
display(pd.DataFrame(found))

# Ambil HANYA bagian yang relevan dari SOP, bukan seluruh dokumen.
def section(doc_id, heading):
    txt=sop_docs[doc_id]['text']
    pattern=rf'(?ms)^({heading})\s+.*?(?=^\d+\.\s+[A-Z]|\Z)'
    match=re.search(pattern,txt)
    return match.group(0).strip() if match else ''

print('SOP sensor, bagian PEMANTAUAN:')
print(section('SOP-PS-002',r'5\.'))

Dokumen: ['SOP-PS-001', 'SOP-PS-002', 'SOP-PS-003']


,doc_id,score,revision,source
0,SOP-PS-002,2,00,SOP-PS-002_Pemantauan_Suhu_dan_Getaran.txt
1,SOP-PS-003,2,00,SOP-PS-003_Pelaporan_dan_Persetujuan_Tindak_La...


SOP sensor, bagian PEMANTAUAN:
5. PEMANTAUAN
5.1 Tinjau kualitas data: data hilang, duplikasi, sensor tidak aktif, dan perubahan mendadak yang berpotensi disebabkan gangguan sensor.
5.2 Bandingkan nilai pengamatan dengan riwayat aset dan dokumen ambang resmi yang berlaku. Dokumen ini tidak menetapkan batas suhu atau getaran universal.
5.3 Bila terdapat penyimpangan, lakukan verifikasi silang terhadap informasi operasi, riwayat gangguan, dan kondisi inspeksi.
5.4 Catat hasil pengukuran dan analisis tren sebelum mengusulkan tindakan.
5.5 Jangan menggunakan output model prediktif sebagai satu-satunya dasar keputusan kelayakan operasi.


### 5. Menjalankan Qwen2.5-0.5B-Instruct secara lokal melalui Ollama

Bagian AI menggunakan model **yang sama** seperti Sesi 4. Tidak perlu Hugging Face atau API key. Perintah berikut memasang Ollama dan mengunduh model langsung ke runtime Colab.

- Jika model sudah terpasang, tidak diunduh ulang.
- Jika instalasi gagal (misalnya jaringan Colab bermasalah), notebook tetap dapat menjalankan alur integrasi; hasil AI akan berstatus `MODEL_UNAVAILABLE`, **bukan jawaban palsu**.
- Inferensi CPU dapat lambat. Latensi diukur agar mahasiswa dapat melihat konsekuensinya.

In [ ]:
import shutil, urllib.request, os, subprocess, time, tarfile

MODEL='qwen2.5:0.5b-instruct'
OLLAMA_URL='http://127.0.0.1:11434/api/generate'
USE_MODEL=True   # Ubah ke False jika hanya ingin menguji integrasi API tanpa inferensi.
MODEL_READY=False

def prepare_ollama():
    """Menjalankan server Ollama lokal dan menarik model sekali."""
    global MODEL_READY
    if not USE_MODEL:
        print('Inferensi dinonaktifkan oleh pengguna.')
        return

    # Bersihkan sisa-sisa binary lama agar terhindar dari conflict
    old_binary = '/usr/local/bin/ollama'
    if os.path.exists(old_binary):
        print('Membersihkan binary Ollama lama...')
        try:
            os.remove(old_binary)
        except Exception:
            subprocess.run(f'rm -f {old_binary}', shell=True)

    # Jika binary 'ollama' tidak ada di PATH, unduh release resmi tarball Linux x64
    if shutil.which('ollama') is None:
        print('Mengunduh paket resmi Ollama via Python...')
        tar_path = '/tmp/ollama.tgz'
        # Menggunakan direct release URL versi spesifik yang stabil dan didukung
        url = 'https://github.com/ollama/ollama/releases/download/v0.3.14/ollama-linux-amd64.tgz'

        # Buka URL secara fleksibel mengikuti redirects
        opener = urllib.request.build_opener(urllib.request.HTTPRedirectHandler)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with opener.open(req) as response, open(tar_path, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)

        print('Mengekstrak paket Ollama secara terprogram...')
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path='/usr/local')

        # Pastikan permission executable terpasang
        os.chmod('/usr/local/bin/ollama', 0o755)

    # Pastikan server Ollama berjalan di latar belakang
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Server Ollama sudah aktif.')
    except Exception:
        print('Memulai server Ollama di background...')
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # Tunggu hingga server siap merespons
        for i in range(30):
            try:
                urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
                print('Server Ollama berhasil merespons!')
                break
            except Exception:
                time.sleep(1)
                if i == 29:
                    raise RuntimeError('Gagal memulai server Ollama setelah 30 detik.')

    print(f'Menarik model {MODEL}...')
    subprocess.run(['ollama', 'pull', MODEL], check=True)
    MODEL_READY = True

try:
    prepare_ollama()
except Exception as exc:
    MODEL_READY = False
    print('Model tidak tersedia:', type(exc).__name__, str(exc)[:220])
print('MODEL_READY =', MODEL_READY)

Membersihkan binary Ollama lama...
Mengunduh paket resmi Ollama via Python...
Mengekstrak paket Ollama secara terprogram...


/tmp/ipykernel_585/2219456404.py:39: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path='/usr/local')


Server Ollama sudah aktif.
Menarik model qwen2.5:0.5b-instruct...
MODEL_READY = True


### 6. Workflow: validasi → API → retrieval → AI → review draft

**Penting:** Model hanya boleh membantu menjelaskan data dan SOP. Aplikasi menentukan aturan akses, status error, sitasi, dan hak persetujuan. **Tidak ada tindakan keselamatan atau work order yang dikirim otomatis.**

In [ ]:
import requests

AUDIT=[]

def generate_ai(prompt):
    if not MODEL_READY:
        return None, 'MODEL_UNAVAILABLE', 0.0
    started=time.perf_counter()
    try:
        res=requests.post(OLLAMA_URL,json={'model':MODEL,'prompt':prompt,'stream':False,
                                'options':{'temperature':0,'num_predict':180,'num_ctx':4096}},timeout=180)
        res.raise_for_status()
        data=res.json()
        return data.get('response','').strip(), None, (time.perf_counter()-started)*1000
    except Exception as exc:
        return None, f'MODEL_ERROR:{type(exc).__name__}', (time.perf_counter()-started)*1000

def run_workflow(asset_id='TR-001',user='teknisi_a',question=QUESTION):
    """Workflow berurutan dengan state dan pemeriksaan error tiap langkah."""
    t0=time.perf_counter()
    trace_id=uuid.uuid4().hex[:10]
    state={'trace_id':trace_id,'asset_id':asset_id,'user':user,'steps':[],
           'status':'started','source_ids':[],'model_response':None,'draft_status':None}
    def step(name,status): state['steps'].append({'step':name,'status':status})
    res=client.get(f'/assets/{asset_id}/sensors/latest',headers={'x-user':user})
    step('sensor_api',str(res.status_code))
    if res.status_code != 200:
        state['status']='SENSOR_API_ERROR'
    else:
        reading=res.json()['reading'];state['reading']=reading
        evidence=retrieve_sops(question,top_k=3)
        state['source_ids']=[item['doc_id'] for item in evidence]
        step('retrieval','ok' if evidence else 'not_found')
        if not evidence:
            state['status']='SOP_NOT_FOUND'
        else:
            # Pilih potongan yang relevan; dua SOP memuat data ukur dan persetujuan.
            excerpts=[]
            for item in evidence:
                docid=item['doc_id']
                fragment=section(docid,r'5\.') or sop_docs[docid]['text'][:1200]
                excerpts.append(f"[{docid}, revisi {item['revision']}]\n{fragment[:1300]}")
            context='\n\n'.join(excerpts)
            prompt=("Anda asisten praktikum pemeliharaan kereta. Gunakan hanya data dan SOP berikut. "
                "Jangan mengarang ambang suhu/getaran atau menyimpulkan kelayakan operasi. "
                "Jika standar ambang belum disediakan, katakan tidak dapat dipastikan. "
                "Jelaskan siapa yang harus menyetujui tindakan. Ringkas dalam 3 kalimat.\n"
                f"PERTANYAAN: {question}\nDATA TERUKUR: {json.dumps(reading,ensure_ascii=False)}\n"
                f"BUKTI SOP:\n{context}\nJAWABAN:")
            response,error,model_ms=generate_ai(prompt)
            state['model_ms']=round(model_ms,1)
            step('model','ok' if error is None else error)
            state['model_response']=response
            # Pemeriksaan sumber dan kewenangan oleh aplikasi (bukan model).
            state['source_ids']=[item['doc_id'] for item in evidence]
            state['citation_valid']=bool(state['source_ids'])
            state['approval_required']=True
            state['draft_status']='PENDING_SUPERVISOR_REVIEW'
            state['status']='READY_FOR_REVIEW' if error is None else 'REVIEW_WITHOUT_AI'
    state['latency_ms']=round((time.perf_counter()-t0)*1000,1)
    AUDIT.append({'trace_id':trace_id,'asset_id':asset_id,'user':user,
                  'status':state['status'],'timestamp':datetime.now(timezone.utc).isoformat(),
                  'latency_ms':state['latency_ms'],'source_ids':','.join(state['source_ids'])})
    return state

result=run_workflow()
print('Status:',result['status'])
print('Sumber:',result['source_ids'])
print('Data sensor:',result.get('reading'))
print('Hasil AI:',result.get('model_response'))
print('Status draft:',result.get('draft_status'))
display(pd.DataFrame(result['steps']))

Status: READY_FOR_REVIEW
Sumber: ['SOP-PS-002', 'SOP-PS-003']
Data sensor: {'event_id': 'evt-1002', 'asset_id': 'TR-001', 'temperature_c': 92.0, 'vibration_mm_s': 3.8, 'timestamp': '2026-11-11T10:00:00+07:00', 'unit': 'C', 'source': 'simulated-sensor'}
Hasil AI: 1. Suhu bearing TR-001: 92.0°C (dalam C).
2. Tidak dapat dipastikan jika suhu tersebut benar-benar sesuai dengan standar ambang.
3. Menyetujui tindakan: Supervisor atau pengguna yang telah mendapat otorisasi.

Penjelasan:
- Suhu 92°C adalah tingkat suhu yang disediakan dalam SOP, tetapi tidak dapat dipastikan jika ini benar-benar sesuai dengan standar ambang.
- Tidak ada data yang menunjukkan bahwa suhu tersebut sesuai dengan standar ambang atau tidak.
- Standar ambang belum disediakan dan tidak bisa dipastikan apakah suhu 92°C sesuai dengan standar ambang.
- Jika SOP telah diterima, supervisor
Status draft: PENDING_SUPERVISOR_REVIEW


,step,status
0,sensor_api,200
1,retrieval,ok
2,model,ok


**Apa yang diamati?** Jalankan `run_workflow(asset_id="TR-002", user="teknisi_a")`: apakah aplikasi tetap memanggil model? Ulangi dengan `asset_id="TR-001"` dan pertanyaan tentang *rem* sehingga tidak ada SOP relevan. Mengapa status berhenti sebelum membuat rekomendasi?

In [ ]:
# Uji failure path yang aman: tidak ada respons AI dari data yang tidak tersedia.
error_trials=[]
for asset_id,user,q in [
    ('TR-002','teknisi_a',QUESTION),
    ('TR-999','supervisor_a',QUESTION),
    ('TR-001','teknisi_a','Bagaimana SOP pemeriksaan rem hidraulik?')]:
    r=run_workflow(asset_id,user,q)
    error_trials.append({'asset_id':asset_id,'user':user,'question':q,
                         'status':r['status'],'model_called':any(s['step']=='model' for s in r['steps'])})
display(pd.DataFrame(error_trials))

,asset_id,user,question,status,model_called
0,TR-002,teknisi_a,"Berapa suhu bearing TR-001, apa yang bisa disi...",SENSOR_API_ERROR,False
1,TR-999,supervisor_a,"Berapa suhu bearing TR-001, apa yang bisa disi...",SENSOR_API_ERROR,False
2,TR-001,teknisi_a,Bagaimana SOP pemeriksaan rem hidraulik?,SOP_NOT_FOUND,False


## Eksperimen 3 — Event-Driven Processing

### 7. Producer, queue, dan consumer

Sensor memproduksi pesan `temperature_updated`. **Queue** menyimpan pesan; **consumer** memvalidasi dan memprosesnya.

Dalam produksi, queue persisten (mis. broker) mempunyai mekanisme delivery dan recovery. Untuk praktikum Colab, `asyncio.Queue` hanya mensimulasikan konsep tersebut **di satu proses Python**, sehingga tidak tahan jika runtime dihentikan.

Aplikasi harus mengenali `event_id` yang sama agar tidak memproses event duplikat dua kali. Kita membuat **draft pemeriksaan administratif** saja, bukan perintah kerja operasional.

In [ ]:
EVENT_LOG=[]
REVIEW_DRAFTS={}
seen_event_ids=set()

async def event_producer(queue,events):
    for event in events:
        await queue.put(event)
    await queue.put(None)  # Penanda semua event telah dikirim.

async def event_consumer(queue):
    while True:
        event=await queue.get()
        if event is None:
            queue.task_done()
            break
        event_id=event.get('event_id')
        if event_id in seen_event_ids:
            EVENT_LOG.append({'event_id':event_id,'status':'DUPLICATE_IGNORED','asset_id':event.get('asset_id')})
            queue.task_done()
            continue
        clean,err=validate_sensor(event)
        if err:
            EVENT_LOG.append({'event_id':event_id,'status':err,'asset_id':event.get('asset_id')})
        elif clean['asset_id'] not in ASSETS:
            EVENT_LOG.append({'event_id':event_id,'status':'ASSET_NOT_FOUND','asset_id':clean['asset_id']})
        else:
            seen_event_ids.add(event_id)
            # Event suhu tidak otomatis mengindikasikan pelanggaran ambang.
            # Draft ini meminta supervisor meninjau DATA, bukan menyatakan kondisi tidak aman.
            REVIEW_DRAFTS[event_id]={'event_id':event_id,'asset_id':clean['asset_id'],
                  'temperature_c':clean['temperature_c'],'timestamp':clean['timestamp'],
                  'status':'PENDING_SUPERVISOR_REVIEW','action':'REVIEW_SENSOR_READING',
                  'approved_by':None}
            EVENT_LOG.append({'event_id':event_id,'status':'DRAFT_CREATED_FOR_REVIEW','asset_id':clean['asset_id']})
        queue.task_done()

async def run_event_demo():
    q=asyncio.Queue()
    events=valid[:3]+[dict(valid[1])]+[raw[-1]]  # 3 event valid + 1 duplikat + 1 invalid
    await asyncio.gather(event_producer(q,events),event_consumer(q))

await run_event_demo()
display(pd.DataFrame(EVENT_LOG))
display(pd.DataFrame(REVIEW_DRAFTS.values()))

,event_id,status,asset_id
0,evt-1001,DRAFT_CREATED_FOR_REVIEW,TR-001
1,evt-1002,DRAFT_CREATED_FOR_REVIEW,TR-001
2,evt-2001,DRAFT_CREATED_FOR_REVIEW,TR-002
3,evt-1002,DUPLICATE_IGNORED,TR-001
4,evt-bad,INVALID_TYPE_OR_TIMESTAMP,TR-001


,event_id,asset_id,temperature_c,timestamp,status,action,approved_by
0,evt-1001,TR-001,89.0,2026-11-11T09:55:00+07:00,PENDING_SUPERVISOR_REVIEW,REVIEW_SENSOR_READING,None
1,evt-1002,TR-001,92.0,2026-11-11T10:00:00+07:00,PENDING_SUPERVISOR_REVIEW,REVIEW_SENSOR_READING,None
2,evt-2001,TR-002,56.0,2026-11-11T10:01:00+07:00,PENDING_SUPERVISOR_REVIEW,REVIEW_SENSOR_READING,None


### 8. Persetujuan supervisor dan audit trail

Draft tidak sama dengan work order. Hanya peran `supervisor` yang boleh memberikan keputusan dalam simulasi. Tidak ada koneksi ke Maintenance Management sungguhan.

In [ ]:
APPROVAL_LOG=[]
ROLES={'teknisi_a':'technician','supervisor_a':'supervisor'}

def review_draft(event_id, reviewer, decision):
    """Kontrol approval sederhana untuk simulasi; hanya keputusan review administratif."""
    if ROLES.get(reviewer) != 'supervisor':
        return {'status':'FORBIDDEN','detail':'Hanya supervisor dapat memutuskan'}
    if decision not in {'APPROVE_REVIEW','REJECT_REVIEW'}:
        return {'status':'INVALID_DECISION'}
    if event_id not in REVIEW_DRAFTS:
        return {'status':'NOT_FOUND'}
    draft=REVIEW_DRAFTS[event_id]
    if draft['status']!='PENDING_SUPERVISOR_REVIEW':
        return {'status':'ALREADY_REVIEWED'}
    draft['status']='REVIEW_APPROVED' if decision=='APPROVE_REVIEW' else 'REVIEW_REJECTED'
    draft['approved_by']=reviewer
    APPROVAL_LOG.append({'event_id':event_id,'reviewer':reviewer,'decision':decision,
                         'timestamp':datetime.now(timezone.utc).isoformat()})
    return {'status':draft['status'],'event_id':event_id}

print('Percobaan teknisi:',review_draft('evt-1002','teknisi_a','APPROVE_REVIEW'))
print('Persetujuan supervisor:',review_draft('evt-1002','supervisor_a','APPROVE_REVIEW'))
print('Persetujuan ulang:',review_draft('evt-1002','supervisor_a','APPROVE_REVIEW'))
display(pd.DataFrame(APPROVAL_LOG))

Percobaan teknisi: {'status': 'FORBIDDEN', 'detail': 'Hanya supervisor dapat memutuskan'}
Persetujuan supervisor: {'status': 'REVIEW_APPROVED', 'event_id': 'evt-1002'}
Persetujuan ulang: {'status': 'ALREADY_REVIEWED'}


,event_id,reviewer,decision,timestamp
0,evt-1002,supervisor_a,APPROVE_REVIEW,2026-11-12T09:01:06.837697+00:00


## 9. Pengujian otomatis sederhana

Perhatikan perbedaan **unit test** (memeriksa satu fungsi) dan **integration test** (memeriksa perpindahan data antar komponen). Tes ini tidak memerlukan hasil generasi model tertentu; model probabilistik tidak sebaiknya diuji dengan satu string jawaban yang harus persis sama.

In [ ]:
# Unit test validasi
assert validate_sensor(readings[1])[1] is None
assert validate_sensor(readings[-1])[1] == 'INVALID_TYPE_OR_TIMESTAMP'
# Integration test endpoint API + otorisasi
assert client.get('/assets/TR-001/sensors/latest',headers={'x-user':'teknisi_a'}).status_code==200
assert client.get('/assets/TR-002/sensors/latest',headers={'x-user':'teknisi_a'}).status_code==403
# Workflow harus berhenti jika sensor gagal.
assert run_workflow('TR-999','supervisor_a')['status']=='SENSOR_API_ERROR'
# Event yang sama diproses satu kali saja.
assert sum(e['status']=='DUPLICATE_IGNORED' for e in EVENT_LOG)==1
# Approval tidak boleh dilakukan oleh teknisi.
assert review_draft('evt-1001','teknisi_a','APPROVE_REVIEW')['status']=='FORBIDDEN'
print('Semua pengujian otomatis lulus.')

Semua pengujian otomatis lulus.


## 10. Ekspor ke CSV untuk Excel

File keluaran: hasil API, ingestion yang ditolak, workflow/audit, event log, dan approval log. Format `UTF-8 with BOM` membantu Excel membaca teks Indonesia dengan benar.

**Hal yang perlu dinilai mahasiswa:** (1) kelengkapan data; (2) status API; (3) ketepatan pemilihan SOP; (4) apakah model menghindari batas suhu yang tidak ada; (5) penanganan duplikasi; (6) kejelasan persetujuan. Untuk evaluasi kualitas jawaban AI, mahasiswa tetap perlu memberi *human score* secara manual.

In [ ]:
def save_csv(name,rows):
    path=OUT/name
    pd.DataFrame(rows).to_csv(path,index=False,encoding='utf-8-sig')
    return path

files_to_export=[
    save_csv('01_api_trials.csv',api_trials),
    save_csv('02_ingestion_rejected.csv',rejected),
    save_csv('03_workflow_audit.csv',AUDIT),
    save_csv('04_event_log.csv',EVENT_LOG),
    save_csv('05_review_drafts.csv',list(REVIEW_DRAFTS.values())),
    save_csv('06_approval_log.csv',APPROVAL_LOG),
]
# Simpan respons model secara terpisah agar dapat dinilai manusia di Excel.
files_to_export.append(save_csv('07_model_result.csv',[{'trace_id':result['trace_id'],
    'asset_id':result['asset_id'],'model_response':result.get('model_response') or '',
    'source_ids':','.join(result['source_ids']),'status':result['status'],
    'human_score_0_2':''}]))
for path in files_to_export: print(path.name)

from zipfile import ZipFile, ZIP_DEFLATED
result_zip=ROOT/'hasil_praktikum_sesi5.zip'
with ZipFile(result_zip,'w',ZIP_DEFLATED) as z:
    for path in files_to_export:z.write(path,arcname=path.name)
print('ZIP hasil:',result_zip)
# Colab: otomatis munculkan unduhan. Jupyter lokal: file tetap tersimpan.
try:
    from google.colab import files
    files.download(str(result_zip))
except ImportError:
    print('Gunakan file ZIP dari folder kerja di atas.')

01_api_trials.csv
02_ingestion_rejected.csv
03_workflow_audit.csv
04_event_log.csv
05_review_drafts.csv
06_approval_log.csv
07_model_result.csv
ZIP hasil: /content/hasil_praktikum_sesi5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Pertanyaan untuk mahasiswa

1. Mengapa pembacaan sensor harus mempunyai `asset_id`, satuan, dan timestamp?
2. Apa bedanya **HTTP 200** dengan data yang benar dan masih baru?
3. Mengapa sistem perlu berhenti jika API sensor gagal atau SOP tidak ditemukan?
4. Mengapa event duplikat perlu dideteksi menggunakan `event_id`?
5. Apakah nilai 92°C dapat digunakan untuk menyatakan bearing tidak aman berdasarkan tiga SOP ini? Jelaskan.
6. Apa perbedaan draft review, approval, dan work order operasional?
7. Bagaimana penerapan `trace_id` membantu penelusuran kesalahan?

**Tugas kecil:** Ubah satu baris di `sensor_readings.csv` (mis. satuan atau timestamp), jalankan ulang Eksperimen 1–3, lalu tulis 5–8 kalimat tentang dampaknya terhadap API, workflow, dan audit log.

### Rujukan
- Chip Huyen, *AI Engineering* — AI stack, RAG/tools, evaluation dan production considerations.
- Miroslaw Staron, *Engineering Generative AI-Based Software* — arsitektur software, API, dan quality assurance.
- Vasundra Srinivasan, *Data Engineering for Multimodal AI* — data ingestion, contracts, idempotency, identity/time alignment.

**Batasan praktik:** FastAPI TestClient dan `asyncio.Queue` berada dalam satu runtime, bukan sistem terdistribusi. Notebook ini adalah demonstrasi konsep integrasi, bukan perangkat keselamatan produksi.